# 토크나이저 학습 + 검증 노트북 (로컬)

SentencePiece **Unigram / NFKC / vocab 32768** 토크나이저를 셀 단위로 실행하며 확인한다.
(vocab 을 10,240 → 32,768 로 올린 근거는 [docs/model_config_review.md](docs/model_config_review.md) §4 — 10k 는 한글 1음절 piece 1,645개뿐이라 byte-fallback 2.3%, 한글 토큰의 45% 가 음절 단위로 쪼개졌다.)

- 커널: 이 리포의 `.venv` 를 선택할 것 (`sentencepiece`, `pandas` 필요)
- 입력: `pair_train.jsonl`의 user/assistant + `pretrain_train.json`의 문자열 배열 (없으면 각각 `<파일명>.tar.xz`에서 자동 해제)
- 대용량 pretrain JSON은 스트리밍으로 읽어 합친 코퍼스를 만든다. 디스크에 코퍼스 저장 공간이 필요하며, SentencePiece는 전체 코퍼스에서 최대 2,000,000줄을 무작위 표본 추출한다.
- 산출물: `tokenizer/spm.model` — 이후 모델 학습이 그대로 사용하며 git 에 커밋한다. `model.py` 의 `vocab_size=32768` 과 일치해야 하고, 데이터 캐시는 vocab과 모델 해시를 기준으로 자동 재생성된다
- 두 데이터셋을 반영하려면 준비 → 코퍼스 추출 → 학습 셀을 다시 실행한다. 기존 모델 검증만 할 때는 코퍼스 추출과 학습을 건너뛸 수 있다.

In [ ]:
# 0) 준비
from pathlib import Path
import json, tarfile, unicodedata
import pandas as pd

from pretrain_merge_json import iter_texts
from tokenizer.train_tokenizer import extract_corpus, train, SPECIAL_TURN_TOKENS

ROOT = Path.cwd()
CORPUS = ROOT / 'cache' / 'tokenizer_corpus.txt'
CORPUS.parent.mkdir(exist_ok=True)
MODEL_PATH = ROOT / 'tokenizer' / 'spm.model'


def ensure_input(filename):
    path = ROOT / filename
    if not path.is_file():
        archive = ROOT / f'{filename}.tar.xz'
        if not archive.is_file():
            raise FileNotFoundError(f'{path} 또는 {archive}가 필요합니다.')
        print(f'압축 해제: {archive.name}', flush=True)
        with tarfile.open(archive, 'r:xz') as tar:
            tar.extractall(ROOT, filter='data')
        if not path.is_file():
            raise FileNotFoundError(f'{archive}에 {filename}이 없습니다.')
    print('입력:', path, f'({path.stat().st_size / 1e6:.0f} MB)')
    return path


jsonl = ensure_input('pair_train.jsonl')
pretrain_json = ensure_input('pretrain_train.json')


In [ ]:
# 1) 두 데이터셋을 한 코퍼스로 추출 (대용량 파일이므로 시간이 걸릴 수 있음)
#    JSON 배열을 통째로 메모리에 올리지 않고 기존 스트리밍 파서를 사용한다.
def extract_training_corpus(pair_path, pretrain_path, corpus_path):
    tmp = corpus_path.with_suffix('.tmp')
    try:
        n_pair = extract_corpus(pair_path, tmp)
        print(f'pair: {n_pair:,}줄', flush=True)
        n_pretrain = 0
        with tmp.open('a', encoding='utf-8') as out:
            for text in iter_texts(pretrain_path):
                text = text.strip()
                if text:
                    out.write(text.replace('\r', ' ').replace('\n', ' ') + '\n')
                    n_pretrain += 1
                    if n_pretrain % 100_000 == 0:
                        print(f'pretrain: {n_pretrain:,}줄 처리 중', flush=True)
        if not n_pair or not n_pretrain:
            raise ValueError('두 데이터셋 모두 비어 있지 않은 학습 텍스트가 필요합니다.')
        tmp.replace(corpus_path)
    finally:
        tmp.unlink(missing_ok=True)
    print(f'pretrain: {n_pretrain:,}줄', flush=True)
    return n_pair + n_pretrain


%time n = extract_training_corpus(jsonl, pretrain_json, CORPUS)
print(f'합친 코퍼스 {n:,}줄, {CORPUS.stat().st_size / 1e6:.0f} MB')

with CORPUS.open(encoding='utf-8') as f:
    for _, line in zip(range(5), f):
        print(' |', line.strip()[:80])


In [ ]:
import os
# 2) SentencePiece Unigram 학습 (수 분 소요, 로그가 아래에 출력된다)
#    num_threads 기본값 = os.cpu_count() -> EM 학습 단계가 모든 코어를 사용한다
%time model_path = train(CORPUS, vocab_size=32768, num_threads=os.cpu_count())
print('저장:', model_path)

In [ ]:
# 3) 로드 + 기본 정보
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file=str(MODEL_PATH))

print('vocab size :', sp.get_piece_size())
print('pad/bos/eos/unk id:', sp.pad_id(), sp.bos_id(), sp.eos_id(), sp.unk_id())
for tok in SPECIAL_TURN_TOKENS:
    print(f'{tok:>16} -> id {sp.piece_to_id(tok)}')
print('\n앞 20개 piece:', [sp.id_to_piece(i) for i in range(20)])

In [ ]:
# 4) 인코딩/디코딩 round-trip 확인
samples = [
    '안녕하세요, 국민건강보험법 제5조를 요약해 주세요.',
    '혈압이 140/90 mmHg 이상이면 고혈압입니다.',
    'The quick brown fox jumps over 13 lazy dogs.',
    '이모지 😊 와 한자 漢字, 특수문자 ㈜·※ 도 깨지지 않아야 한다.',
]
for text in samples:
    ids = sp.encode(text)
    decoded = sp.decode(ids)
    ok = decoded == unicodedata.normalize('NFKC', text)   # NFKC 정규화 후 일치해야 정상
    print(f"[{'OK ' if ok else 'DIFF'}] {len(ids):3d} tokens | {text}")
    print('      pieces:', sp.encode(text, out_type=str))
    if not ok:
        print('      decoded:', decoded)

In [ ]:
# 5) 챗 템플릿 인코딩 확인 - 학습 시 실제로 쓰는 형태 (mask=1 구간만 손실 계산)
from data import encode_sample

ids, mask = encode_sample(sp, '감기에 걸렸을 때 어떻게 해야 하나요?', '충분한 휴식과 수분 섭취가 중요합니다.')
df = pd.DataFrame({'id': ids, 'piece': [sp.id_to_piece(i) for i in ids], 'loss_mask': mask})
print(f'총 {len(ids)} tokens, 손실 계산 대상 {sum(mask)} tokens')
df.T

In [ ]:
# 6) 대화 데이터(pair_train.jsonl) 토큰 통계 - 파일 전체에서 등간격 표본 4,000개
#    주의: pair_train.jsonl 은 소스 파일명 순으로 이어 붙인 것(셔플 안 됨)이라 "앞 N줄"만 보면
#    KoAlpaca 한 소스만 보게 된다. 전체 분포(전수 집계)는 docs/seq_len_review.md 참조.
N_SAMPLE = 4000
n_lines = sum(1 for _ in jsonl.open(encoding='utf-8'))
stride = max(n_lines // N_SAMPLE, 1)

rows = []
with jsonl.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % stride:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        ids, mask = encode_sample(sp, obj.get('user', ''), obj.get('assistant', ''))
        rows.append({'tokens': len(ids), 'prompt': len(ids) - sum(mask), 'answer': sum(mask),
                     'chars': len(obj.get('user', '')) + len(obj.get('assistant', ''))})
stats = pd.DataFrame(rows)
stats['chars_per_token'] = stats['chars'] / stats['tokens']
print(f'{n_lines:,}줄 중 {len(stats):,}개 표본 (stride {stride})')
print(stats.describe(percentiles=[.5, .9, .95, .99]).round(1))

for L in (512, 1024, 2048):
    print(f'seq_len {L:5d} 이내 샘플: {(stats.tokens <= L).mean() * 100:5.1f}%')
print(f"\n=> 학습 seq_len 2048: p99 {stats.tokens.quantile(.99):.0f} tokens, 초과 {(stats.tokens > 2048).mean() * 100:.2f}%"
      f" | 답변 p90 {stats.answer.quantile(.9):.0f} tokens (추론 max_new_tokens 512 이상 권장)"
      f" | 평균 압축률 {stats['chars_per_token'].mean():.2f} chars/token")

In [ ]:
# 7) vocab 살펴보기 - 어떤 조각들이 학습됐는지
pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]

longest = sorted(pieces, key=len, reverse=True)[:20]
print('가장 긴 piece 20개:')
for piece in longest:
    print('  ', piece)

n_byte = sum(piece.startswith('<0x') for piece in pieces)
print(f'\nbyte-fallback piece: {n_byte}개 (256개면 정상)')
print('한국어 piece 예시:', [p for p in pieces[100:3000] if any("가" <= c <= "힣" for c in p)][:30])